# IVP: Euler, RK4 y RK45 (Dormand–Prince) — solo Python estándar

Se integra **el mismo** problema de valor inicial en $[t_0,t_f]$ con tres métodos distintos para **comparar precisión** (error frente a solución exacta) y coste (evaluaciones de $f$).

**Restricción:** no se usan NumPy, SciPy ni `torch`; solo `math` y tipos básicos (listas, bucles). Las fórmulas están escritas explícitamente en el código a partir de las tablas de Butcher clásicas.


## Problema modelo

Elegimos un escalar con solución cerrada:
\[
y'(t) = -\,y(t) + \sin(t), \qquad y(0)=0.
\]
Solución exacta (factor integrante $e^{t}$):
\[
y(t)=\frac{1}{2}\bigl(\sin t - \cos t + e^{-t}\bigr).
\]
Comprobación: $y(0)=\frac{1}{2}(0-1+1)=0$. Intervalo de integración $[0,\,3]$.


## 1. Método de Euler explícito

Dada la malla $t_{n+1}=t_n+h$ y la aproximación $y_n\approx y(t_n)$, el **Euler explícito** es
\[
y_{n+1} = y_n + h\,f(t_n,y_n).
\]
Es un método de Runge–Kutta de **una etapa** (peso 1 sobre $k_1=f(t_n,y_n)$). **Orden global** típico $O(h)$.

**Ventaja:** trivial. **Desventaja:** con $h$ grande el error es mayor que en RK de orden alto.


In [1]:
from __future__ import annotations

import math


def f_ivp(t: float, y: float) -> float:
    # y' = -y + sin(t)
    return -y + math.sin(t)


def y_exacta(t: float) -> float:
    return 0.5 * (math.sin(t) - math.cos(t) + math.exp(-t))


def euler(f, t0: float, tf: float, h: float, y0: float):
    # Euler explícito: lista (t, y) y nfev
    nfev = 0
    out = [(t0, y0)]
    t, y = t0, y0
    while t < tf - 1e-15:
        hh = min(h, tf - t)
        k1 = f(t, y)
        nfev += 1
        y = y + hh * k1
        t = t + hh
        out.append((t, y))
    return out, nfev


T0, TF = 0.0, 3.0
Y0 = 0.0
H_EULER = 0.04

sol_eu, nfe_eu = euler(f_ivp, T0, TF, H_EULER, Y0)
err_eu = abs(sol_eu[-1][1] - y_exacta(TF))
print("Euler")
print("  paso h =", H_EULER, "| pasos =", len(sol_eu) - 1, "| nfev =", nfe_eu)
print("  y_aprox(tf) =", sol_eu[-1][1])
print("  y_exact(tf) =", y_exacta(TF))
print("  |error|     =", err_eu)


Euler
  paso h = 0.04 | pasos = 75 | nfev = 75
  y_aprox(tf) = 0.599439367265021
  y_exact(tf) = 0.5904497865140882
  |error|     = 0.008989580750932813


## 2. Runge–Kutta clásico de orden 4 (RK4)

Un paso de longitud $h$ del **RK4** estándar:
\[
\begin{aligned}
k_1 &= f(t_n,\,y_n),\\
k_2 &= f\!\left(t_n+\tfrac{h}{2},\,y_n+\tfrac{h}{2}k_1\right),\\
k_3 &= f\!\left(t_n+\tfrac{h}{2},\,y_n+\tfrac{h}{2}k_2\right),\\
k_4 &= f\!\left(t_n+h,\,y_n+h k_3\right),\\
y_{n+1} &= y_n + \frac{h}{6}\bigl(k_1+2k_2+2k_3+k_4\bigr).
\end{aligned}
\]
**Orden global** 4 en problemas suaves. **4 evaluaciones** de $f$ por paso.


In [2]:
def rk4_paso(f, t: float, y: float, h: float):
    k1 = f(t, y)
    k2 = f(t + 0.5 * h, y + 0.5 * h * k1)
    k3 = f(t + 0.5 * h, y + 0.5 * h * k2)
    k4 = f(t + h, y + h * k3)
    y_new = y + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return y_new, 4


def rk4_fijo(f, t0: float, tf: float, h: float, y0: float):
    out = [(t0, y0)]
    nfev = 0
    t, y = t0, y0
    while t < tf - 1e-15:
        hh = min(h, tf - t)
        y, nf = rk4_paso(f, t, y, hh)
        nfev += nf
        t += hh
        out.append((t, y))
    return out, nfev


H_RK4 = 0.25

sol_r4, nfe_r4 = rk4_fijo(f_ivp, T0, TF, H_RK4, Y0)
err_r4 = abs(sol_r4[-1][1] - y_exacta(TF))
print("RK4")
print("  paso h =", H_RK4, "| pasos =", len(sol_r4) - 1, "| nfev =", nfe_r4)
print("  |error| en tf =", err_r4)


RK4
  paso h = 0.25 | pasos = 12 | nfev = 48
  |error| en tf = 1.8118839788749952e-05


## 3. RK45 = Dormand–Prince 5(4) embebido con paso adaptativo

Los métodos **embebidos** calculan dos aproximaciones de órdenes distintos con las **mismas** pendientes $k_i$; la diferencia estima el **error local**.

El **Dormand–Prince** (base de `ode45` y `RK45` en SciPy) usa 7 etapas en la tabla; con **FSAL**, en la práctica son **6 evaluaciones nuevas** por paso aceptado.

\[
y^{(5)}_{n+1} = y_n + h\sum_{j=1}^{7} b^{(5)}_j k_j,
\qquad
y^{(4)}_{n+1} = y_n + h\sum_{j=1}^{7} b^{(4)}_j k_j,
\qquad
\text{err} \approx \bigl|\,y^{(5)}_{n+1} - y^{(4)}_{n+1}\,\bigr|.
\]

Se escala con $\text{sc}=\text{atol}+\text{rtol}\cdot\max(|y_n|,|y^{(5)}|)$ y se acepta o rechaza el paso; $h$ se ajusta con factores de seguridad estándar.

Los $a_{ij}$, $b^{(5)}_j$, $b^{(4)}_j$ y $c_i$ del código coinciden con la tabla de Dormand–Prince (1980).


In [3]:
def dormand_prince_paso(f, t: float, y: float, h: float):
    c = [0.0, 1.0 / 5.0, 3.0 / 10.0, 4.0 / 5.0, 8.0 / 9.0, 1.0, 1.0]

    a21 = 1.0 / 5.0
    a31, a32 = 3.0 / 40.0, 9.0 / 40.0
    a41, a42, a43 = 44.0 / 45.0, -56.0 / 15.0, 32.0 / 9.0
    a51 = 19372.0 / 6561.0
    a52 = -25360.0 / 2187.0
    a53 = 64448.0 / 6561.0
    a54 = -212.0 / 729.0
    a61 = 9017.0 / 3168.0
    a62 = -355.0 / 33.0
    a63 = 46732.0 / 5247.0
    a64 = 49.0 / 176.0
    a65 = -5103.0 / 18656.0
    a71, a72, a73, a74, a75, a76 = (
        35.0 / 384.0,
        0.0,
        500.0 / 1113.0,
        125.0 / 192.0,
        -2187.0 / 6784.0,
        11.0 / 84.0,
    )

    b5 = [35.0 / 384.0, 0.0, 500.0 / 1113.0, 125.0 / 192.0, -2187.0 / 6784.0, 11.0 / 84.0, 0.0]
    b4 = [
        5179.0 / 57600.0,
        0.0,
        7571.0 / 16695.0,
        393.0 / 640.0,
        -92097.0 / 339200.0,
        187.0 / 2100.0,
        1.0 / 40.0,
    ]

    k = [0.0] * 7
    nfev = 0

    k[0] = f(t, y)
    nfev += 1

    y2 = y + h * (a21 * k[0])
    k[1] = f(t + c[1] * h, y2)
    nfev += 1

    y3 = y + h * (a31 * k[0] + a32 * k[1])
    k[2] = f(t + c[2] * h, y3)
    nfev += 1

    y4 = y + h * (a41 * k[0] + a42 * k[1] + a43 * k[2])
    k[3] = f(t + c[3] * h, y4)
    nfev += 1

    y5 = y + h * (a51 * k[0] + a52 * k[1] + a53 * k[2] + a54 * k[3])
    k[4] = f(t + c[4] * h, y5)
    nfev += 1

    y6 = y + h * (a61 * k[0] + a62 * k[1] + a63 * k[2] + a64 * k[3] + a65 * k[4])
    k[5] = f(t + c[5] * h, y6)
    nfev += 1

    y7 = y + h * (a71 * k[0] + a72 * k[1] + a73 * k[2] + a74 * k[3] + a75 * k[4] + a76 * k[5])
    k[6] = f(t + c[6] * h, y7)
    nfev += 1

    y5_final = y + h * sum(b5[j] * k[j] for j in range(7))
    y4_final = y + h * sum(b4[j] * k[j] for j in range(7))
    err_est = abs(y5_final - y4_final)

    return t + h, y5_final, err_est, nfev


def integrar_dp45(f, t0: float, tf: float, y0: float, h0: float, atol: float, rtol: float, h_min: float):
    t, y = t0, y0
    h = h0
    out = [(t, y)]
    nfev_total = 0
    pasos_ok = pasos_rech = 0

    while t < tf - 1e-14:
        if h < h_min:
            raise RuntimeError("h llegó a h_min")

        hh = min(h, tf - t)
        t_new, y_new, err, nf = dormand_prince_paso(f, t, y, hh)
        nfev_total += nf

        sc = atol + rtol * max(abs(y), abs(y_new))
        if sc < 1e-30:
            sc = 1e-30
        err_rel = err / sc

        if err_rel <= 1.0 or hh <= h_min * (1.0 + 1e-12):
            t, y = t_new, y_new
            out.append((t, y))
            pasos_ok += 1
            if err_rel > 0:
                fac = 0.9 * (1.0 / err_rel) ** 0.2
            else:
                fac = 5.0
            fac = max(0.2, min(fac, 5.0))
            h = min(h * fac, tf - t) if t < tf - 1e-14 else hh
        else:
            pasos_rech += 1
            h = max(h_min, 0.5 * hh)

    return out, nfev_total, pasos_ok, pasos_rech


ATOL, RTOL = 1e-9, 1e-6
H0_DP = 0.3

sol_dp, nfe_dp, ok_dp, rej_dp = integrar_dp45(f_ivp, T0, TF, Y0, H0_DP, ATOL, RTOL, h_min=1e-12)
err_dp = abs(sol_dp[-1][1] - y_exacta(TF))

print("RK45 (Dormand–Prince adaptativo)")
print("  atol, rtol =", ATOL, RTOL, "| h inicial =", H0_DP)
print("  pasos aceptados =", ok_dp, "| rechazados =", rej_dp, "| nfev =", nfe_dp)
print("  |error| en tf =", err_dp)


RK45 (Dormand–Prince adaptativo)
  atol, rtol = 1e-09 1e-06 | h inicial = 0.3
  pasos aceptados = 16 | rechazados = 4 | nfev = 140
  |error| en tf = 1.8115366762216922e-07


## 4. Tabla comparativa

Mismo IVP, mismo $t_f$. Se muestran **error absoluto** en $t_f$ y **número de evaluaciones de $f$** (coste típico). Ajustá `H_EULER`, `H_RK4`, `ATOL`/`RTOL` en las celdas anteriores para experimentar.


In [4]:
print("--- Resumen ---")
print(f"{'Método':<28} {'|error| en tf':>16} {'# evaluaciones f':>18}")
print("-" * 64)
print(f"{'Euler (h fijo)':<28} {err_eu:>16.3e} {nfe_eu:>18d}")
print(f"{'RK4 (h fijo)':<28} {err_r4:>16.3e} {nfe_r4:>18d}")
print(f"{'RK45 DP adaptativo':<28} {err_dp:>16.3e} {nfe_dp:>18d}")
print(f"\nSolución exacta y(tf) = {y_exacta(TF):.12f}")


--- Resumen ---
Método                          |error| en tf   # evaluaciones f
----------------------------------------------------------------
Euler (h fijo)                      8.990e-03                 75
RK4 (h fijo)                        1.812e-05                 48
RK45 DP adaptativo                  1.812e-07                140

Solución exacta y(tf) = 0.590449786514
